In [27]:
import pandas as pd

file_path = '../data/2024_Finaccess_Publicdata.xlsx'

# Load the full survey dataset (this is the heavy operation that takes a few minutes)
#df = pd.read_excel(file_path, sheet_name='2024_Finaccess_Publicdata')

print("Dataset loaded successfully!")
print("Shape:", df.shape)

Dataset loaded successfully!
Shape: (20871, 3816)


In [28]:
# Convert all object/mixed-type columns to strings to satisfy Parquet requirements
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].astype(str)

# Now save to Parquet successfully
#df.to_parquet('data/finaccess_2024.parquet', index=False)
print("Dataset successfully saved to Parquet!")

Dataset successfully saved to Parquet!


In [29]:
# For all future sessions, you can skip the Excel load and run this in 3 seconds:
df = pd.read_parquet('data/finaccess_2024.parquet')

print("Columns:", df.columns.tolist()[:20])
display(df.head())

Columns: ['interview__key', 'interview__id', 'county', 'A07', 'A08', 'A10', 'Interview_Status', 'A9i', 'Z1', 'A13', 'A12ii_length', 'A14iv', 'A14vi', 'FHH_Spouse', 'A15a', 'A15i', 'A18', 'A19', 'A14i', 'A14ii']


,interview__key,interview__id,county,A07,A08,A10,Interview_Status,A9i,Z1,A13,...,sacco_redress,mobilemoney_redress,mobilebank_redress,not_registered_mmoney_24,using_someone_acc,insurance_including_NHIF_use,All_Insurance_excluding_NHIF_use,PWD,Latitude,Longitude
0,21-61-43-44,00005e63e7424aa19e63adb071955233,Garissa,225,Urban,1,Household found with competent respondent,Access granted,completed,Male,...,nan,nan,nan,No,nan,Never used,Never used,Without Disability,-0.435423,39.636586
1,87-78-54-60,000718d26f5942998522033a66594b4c,Garissa,223,Urban,3,Household found with competent respondent,Access granted,completed,Male,...,nan,nan,nan,No,nan,Never used,Never used,Without Disability,0.058794,40.305006
2,32-71-19-84,0008721681d2468aa1ef456bba4475dc,Busia,1514,Urban,3,Household found with competent respondent,Access granted,completed,Female,...,nan,nan,nan,No,nan,Never used,Never used,Without Disability,0.636836,34.277390
3,48-18-78-99,00099e961d49470585c1a7f49fa3b7fb,Kiambu,796,Urban,3,Household found with competent respondent,Access granted,completed,Male,...,nan,nan,nan,No,nan,Never used,Never used,Without Disability,-1.251917,36.719076
4,50-82-98-89,000bb72e959b4dd48443da902c01e920,Murang'a,763,Urban,16,Household found with competent respondent,Access granted,completed,Female,...,nan,nan,nan,No,Yes,Never used,Never used,Without Disability,-0.795820,37.131085


In [30]:
# Load the Variables sheet (the data dictionary)
dictionary_df = pd.read_excel(file_path, sheet_name='Variables')

# Search for columns related to our key themes
# We will look for keywords in the variable labels/descriptions
keywords = ['income', 'farm', 'agriculture', 'mobile money', 'M-Pesa']

# Filter the dictionary to show only rows containing our keywords
search_results = dictionary_df[
    dictionary_df.astype(str).apply(lambda x: x.str.contains('|'.join(keywords), case=False)).any(axis=1)
]

print("Found", len(search_results), "relevant variables.")
display(search_results.head(20))

Found 214 relevant variables.


,position,name,type,isnumeric,format,vallab,varlab
73,74,B1L__5,byte,1,%12.0g,B1L__5,"B1L__5:Machinery, vehicles, equipment, tools f..."
97,98,B3A__1,byte,1,%12.0g,NaN,"B3A. Farming (crops, keeping livestock, fishin..."
110,111,B3Ai,strL,0,%477s,NaN,B3Ai. Income Sources- Other
120,121,B3D__2_1,byte,1,%12.0g,NaN,B3D__2.1: B3D. Received Payments:Mobile money ...
121,122,B3D__3_1,byte,1,%12.0g,NaN,B3D__3.1: B3D. Received Payments:Mobile money ...
135,136,B3D__2_2,byte,1,%12.0g,NaN,B3D__2.2: B3D. Received Payments:Mobile money ...
136,137,B3D__3_2,byte,1,%12.0g,NaN,B3D__3.2: B3D. Received Payments:Mobile money ...
150,151,B3D__2_3,byte,1,%12.0g,NaN,B3D__2.3: B3D. Received Payments:Mobile money ...
151,152,B3D__3_3,byte,1,%12.0g,NaN,B3D__3.3: B3D. Received Payments:Mobile money ...
165,166,B3D__2_4,byte,1,%12.0g,NaN,B3D__2.4: B3D. Received Payments:Mobile money ...


In [31]:
vars_df = pd.read_excel('../data/2024_Finaccess_Publicdata.xlsx', sheet_name='Variables')

keywords = ['loan', 'credit', 'borrow', 'debt', 'repay', 'default', 
            'arrears', 'asset', 'machinery', 'equipment', 'solar', 
            'mobile money', 'lend', 'installment']

mask = vars_df['varlab'].str.contains('|'.join(keywords), case=False, na=False) | \
       vars_df['name'].str.contains('|'.join(keywords), case=False, na=False)

relevant_vars = vars_df[mask][['name', 'varlab']]
print(relevant_vars.to_string())

                                  name                                                                            varlab
63                                B1Hi                                                      B1Hi. Source of loans/credit
72                              B1L__4  B1L__4:Machinery, vehicles, equipment, tools, building for business/self-employm
73                              B1L__5     B1L__5:Machinery, vehicles, equipment, tools for farming or raising livestock
102                             B3A__6                                        B3A. Renting, land, house/rooms, equipment
120                           B3D__2_1  B3D__2.1: B3D. Received Payments:Mobile money (Send money) (M-Pesa, Airtel Money
121                           B3D__3_1  B3D__3.1: B3D. Received Payments:Mobile money business wallet (e.g. Pochi la bia
126                           B3D__8_1                       B3D__8.1: B3D. Received Payments:Credit cards / debit cards
135                           B3

In [32]:
pd.set_option('display.max_rows', None)
relevant_vars.to_csv('data/relevant_vars.csv', index=False)

In [33]:
import os
os.makedirs('data', exist_ok=True)

In [34]:
import pandas as pd
vars_df = pd.read_excel('../data/2024_Finaccess_Publicdata.xlsx', sheet_name='Variables')

keywords = ['loan', 'credit', 'borrow', 'debt', 'repay', 'default', 
            'arrears', 'asset', 'machinery', 'equipment', 'solar', 
            'mobile money', 'lend', 'installment', 'food', 'income']

mask = vars_df['varlab'].str.contains('|'.join(keywords), case=False, na=False) | \
       vars_df['name'].str.contains('|'.join(keywords), case=False, na=False)

relevant_vars = vars_df[mask][['name', 'varlab']]
pd.set_option('display.max_rows', None)
relevant_vars.to_csv('data/relevant_vars.csv', index=False)
print("Saved relevant_vars.csv successfully!")

Saved relevant_vars.csv successfully!


In [35]:
b1_vars = vars_df[vars_df['name'].str.startswith('B1', na=False)][['name', 'varlab']]
b1_vars.to_csv('data/b1_vars.csv', index=False)
print(f"Saved {len(b1_vars)} B1 variables to data/b1_vars.csv")

Saved 41 B1 variables to data/b1_vars.csv


In [36]:
repay_keywords = ['repay', 'overdue', 'behind', 'missed', 'struggl', 'difficult', 'arrear', 'default', 'unable to pay']
repay_mask = vars_df['varlab'].str.contains('|'.join(repay_keywords), case=False, na=False)
repay_vars = vars_df[repay_mask][['name', 'varlab']]
print(repay_vars.to_string())

                                 name                                                                            varlab
36                               A24i                                                            A24i Difficulty seeing
37                              A24ii                                                           A24i Difficulty hearing
38                             A24iii                                                           A24i Difficulty walking
39                              A24iv                                                    A24i Difficulty with cognition
40                               A24v                                                     A24i Difficulty with selfcare
41                              A24vi                                               A24i Difficulty with  communicating
52                               B1E1                                                B1E-Unable to pay expenses Housing
53                               B1E2   

In [38]:
target_col = 'E2E'

# 1. Show the full breakdown
print("Class Balance for E2E (Loan Default/Delinquency status):")
print(df[target_col].value_counts(dropna=False))

# 2. Filter out the "nan" strings and the "Don't know" answers
valid_responses = [
    'I did not default/always pay on time', 
    'Paid late/Missed a payment/Paid less', 
    "Didn't pay at all"
]

# 3. Create the clean modeling dataframe
modeling_df = df[df[target_col].isin(valid_responses)].copy()

print(f"\nFinal clean active borrowers for the model: {modeling_df.shape[0]}")

Class Balance for E2E (Loan Default/Delinquency status):
E2E
nan                                     7720
I did not default/always pay on time    6160
Paid late/Missed a payment/Paid less    5015
Didn't pay at all                       1936
Donâ€™t know                              37
Refused to Answer                          3
Name: count, dtype: int64

Final clean active borrowers for the model: 13111


In [39]:
# 1. Create the binary target: 
# 0 = Always paid on time, 1 = Late/Missed/Defaulted
modeling_df['defaulted'] = modeling_df['E2E'].apply(
    lambda x: 0 if x == 'I did not default/always pay on time' else 1
)

print("--- Binary Target Balance ---")
print(modeling_df['defaulted'].value_counts(normalize=True).round(3))
print("\n")

# 2. Cross-tabulate against specific loan products to find risk drivers
loan_types = {
    'C1_15': 'Mobile Banking Loan',
    'C1_19': 'Digital App Loan',
    'C1_17': 'Sacco Loan'
}

for col, name in loan_types.items():
    if col in modeling_df.columns:
        print(f"--- Default Rate by: {name} ({col}) ---")
        # normalize='index' shows the percentage breakdown row-by-row
        crosstab = pd.crosstab(modeling_df[col], modeling_df['defaulted'], normalize='index').round(3)
        print(crosstab)
        print("\n")
    else:
        print(f"Column {col} not found in current dataframe.\n")

--- Binary Target Balance ---
defaulted
1    0.53
0    0.47
Name: proportion, dtype: float64


--- Default Rate by: Mobile Banking Loan (C1_15) ---
defaulted          0      1
C1_15                      
Currently use  0.324  0.676
Never used     0.494  0.506
Not stated     1.000  0.000
Used to use    0.463  0.537


--- Default Rate by: Digital App Loan (C1_19) ---
defaulted          0      1
C1_19                      
Currently use  0.228  0.772
Never used     0.479  0.521
Used to use    0.430  0.570


--- Default Rate by: Sacco Loan (C1_17) ---
defaulted          0      1
C1_17                      
Currently use  0.631  0.369
Never used     0.455  0.545
Used to use    0.566  0.434




In [40]:
# Get counts AND percentages together, for every C1 loan-type column
c1_cols = [c for c in modeling_df.columns if c.startswith('C1_')]
results = []

for col in c1_cols:
    # Calculate mean (default rate) and count (n)
    counts = modeling_df.groupby(col)['defaulted'].agg(['mean', 'count']).round(3)
    counts.columns = ['default_rate', 'n']
    counts['loan_type'] = col
    
    # Rename the index column so it's clear what the answer was
    counts.index.name = 'usage_status'
    results.append(counts.reset_index())

# Combine and save
full_results = pd.concat(results, ignore_index=True)
full_results.to_csv('data/default_rate_by_loan_type.csv', index=False)

# Print the full table
print(full_results.to_string())

                            usage_status  default_rate      n       loan_type
0                          Currently use         0.539  11687           C1_10
1                             Never used         0.426    913           C1_10
2                            Used to use         0.519    511           C1_10
3                          Currently use         0.575   4400           C1_11
4                             Never used         0.488   7037           C1_11
5                            Used to use         0.591   1674           C1_11
6                          Currently use         0.484   3718           C1_12
7                             Never used         0.533   7465           C1_12
8                            Used to use         0.607   1928           C1_12
9                    Absa Bank Kenya Plc         0.391     69  C1_12ai_CHK__0
10                   Access Bank Limited         0.500     12  C1_12ai_CHK__0
11       African Banking Corporation Ltd         0.000      1  C

In [41]:
import pandas as pd
import re

# 1. Load the variables dictionary (if not already loaded)
vars_df = pd.read_excel('../data/2024_Finaccess_Publicdata.xlsx', sheet_name='Variables')

# 2. Only keep columns matching the clean pattern C1_<number>
main_c1_cols = [c for c in modeling_df.columns if re.fullmatch(r'C1_\d+', c)]
print(f"Found {len(main_c1_cols)} main loan-type columns:\n")

# 3. Cross-reference with the Variables sheet to get readable names
c1_labels = vars_df[vars_df['name'].isin(main_c1_cols)][['name', 'varlab']]
print("--- Loan Category Dictionary ---")
print(c1_labels.to_string())
print("\n")

# 4. Re-run the default-rate table on the clean list
results = []
for col in main_c1_cols:
    counts = modeling_df.groupby(col)['defaulted'].agg(['mean', 'count']).round(3)
    counts.columns = ['default_rate', 'n']
    counts['loan_type'] = col
    
    # Optional: rename index so it says 'usage_status' like before
    counts.index.name = 'usage_status'
    results.append(counts.reset_index())

full_results = pd.concat(results, ignore_index=True)
full_results.to_csv('data/default_rate_by_loan_type_clean.csv', index=False)

print("--- Cleaned Default Rates ---")
print(full_results.to_string())

Found 41 main loan-type columns:

--- Loan Category Dictionary ---
      name                                                                            varlab
285  C1_10                                                 C1-10. Registered on Mobile money
286  C1_11                                                  C1_11. Registered on Mobile bank
287  C1_12                             C1_12. Registered on a bank/microfinance Bank account
346   C1_2                                     C1-2. Savings through mobile banking Platform
347   C1_3                               C1-3. Savings/keeping through mobile money provider
348   C1_4            C1-4. Savings at a Sacco / Savings and Credit Cooperative organisation
349   C1_5                                                          C1-5. Savings at a chama
350   C1_6                                     C1-6.   Savings with a group/group of friends
351   C1_7                                 C1-7. Savings given to a family or friend to keep
352